### **Porn Detection**
Машинное обучение

Подключим требуемые библеотеки для выполнения работы. ***pandas*** - для считывания данных из csv файлов и работой с ними в виде фреймов (таблиц).
***sklearn*** - для деления на обучающую и тестовую выборки, вычисления метрик модели, векторизации текстовых данных и самой обучающейся модели.
***nltk*** - для токенизации текста и удаления стоп-слов.

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

Считываем тестовые и тренировочные данные и убираем из них строки с NaN значениями

In [17]:
result_test = pd.read_csv("test.csv")
train = pd.read_csv("train.csv")
train = train.dropna()
result_test = result_test.dropna()

Соединим две текстовые колонки url и title в одну text, для более удобной обработки

In [18]:
train["text"] = train["url"] + ' ' + train["title"]
result_test["text"] = result_test["url"] + ' ' + result_test["title"]

Делим выборку на тренировчую и тестовую, на которой будем проверять модель

In [19]:
X_train, X_test, y_train, y_test = train_test_split(train[["text"]], train["label"], test_size=0.2, random_state=42)

Скачиваем необходимые данные для nltk и создаем множество стоп-слов

In [ ]:
nltk.download('punkt_tab')
nltk.download('stopwords')

stop_words = set(stopwords.words('russian', 'english'))

Создаем функцию токенайзера, который преобразует текст к нижнему регистру, токенизирует его и удаляет стоп-слова

In [21]:
def tokenizer(text):
    text = text.lower()
    tokens = word_tokenize(text)
    filtered_words = [word for word in tokens if word not in stop_words]
    
    return ' '.join(filtered_words)

Создаем объект **vectorizer**, с помощью которого векторизируем текст с помощью TF-IDF с изменяющимся значением **ngram** и **features**, изменяя которые обнаружим 
наиболее подходящие гиперпараметры. **ngram** - чтобы учитывать сочетания слов, а не только одного, а **features** отвечает за размерность фич

In [ ]:
'''
maxf = [0, 0, 0]
for features in [5000, 10000, 20000, 30000, 40000]:
    for ngram in [2, 3, 4]:
        vectorizer = TfidfVectorizer(ngram_range=(1, ngram), max_features=features, tokenizer=tokenizer)
        X_train_vec = vectorizer.fit_transform(X_train["text"])
        X_test_vec = vectorizer.transform(X_test["text"])
        X_train_vec.shape
        model = LogisticRegression(penalty="l2")
        model.fit(X_train_vec, y_train)
        predicted = model.predict(X_test_vec)
        if (f1_score(y_test, predicted) > maxf[0]):
            maxf[0] = f1_score(y_test, predicted)
            maxf[1] = ngram
            maxf[2] = features
        print(maxf)
'''
#[0.9683764563474051, 4, 30000]

Наиболее подходящие гиперпараметры будем использвоать для создания объекта векторизации текста, это: размер **ngram** от 1 до 4 и количество **features** 30000

In [22]:
vectorizer = TfidfVectorizer(ngram_range = (1, 4), max_features=30000, tokenizer=tokenizer)

С помощью **fit_transform** обучаем наш объекст векторизации, строим словарь **ngram** и их весов, преобразует текст в матрицу признаков, где строка - один документ, колонка - одна **ngram**
С помощью **transform**, используя уже построенный словарь преобразовываем данные для теста в те же самые признаки.

In [23]:
X_train_vec = vectorizer.fit_transform(X_train["text"])
X_test_vec = vectorizer.transform(X_test["text"])

Обучаем модель логистической регресиии - бинарный классификатор. И счиатем метрики на тестовой выборке

In [24]:
model = LogisticRegression(penalty="l2")
model.fit(X_train_vec, y_train)
predicted = model.predict(X_test_vec)

print(f"F1 метрика: {f1_score(y_test, predicted)}")

F1 метрика: 0.9683764563474051


Применяем обученный векторайзер для тестовых данных (не имеющих таргета ), строя из них матрицу признаков, и предсказываем метки уже обученной моделью бинарной классификации. 

In [25]:
X_test_vec = vectorizer.transform(result_test["text"])
predicted_labels = model.predict(X_test_vec)
result_test['label'] = predicted_labels

Формируем результирующий csv файл, с ID и label

In [26]:
result_test = result_test[['ID', 'label']]
print(result_test[['ID', 'label']].head())
result_test.to_csv("done.csv", index=False)

       ID  label
0  135309      0
1  135310      0
2  135311      0
3  135312      1
4  135313      0
